In [1]:
from ipynb.fs.full. magic_flow_model import *
from ipynb.fs.full. data_loader import *
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, CosineAnnealingLR
import numpy as np
import math
import matplotlib.pyplot as plt

## Directory Setup

In [2]:
NUM_CLASSES = 6

base_path = os.getcwd()
print("Base path:", base_path)

# Define some additions
data_path = os.path.join(base_path, "datasets/modality_data6")
model_folder = os.path.join(base_path, "checkpoints")

Base path: /home/almalinux/data/jupyter_notebooks/cNF_folder/MAGIC-Flow/generation


## Model Initialization

In [3]:
filters_per_layer = [
    (16, 16, 32), (16, 16, 32), (16, 32, 32),  (16, 32, 32), (16, 32, 32),
    # 
    (16, 32, 32), (16, 32, 64), (16, 32, 64),
    # 
    (16, 32, 64), (16, 32, 64), (16, 64, 64),
    # 
    (32, 64, 64), (32, 64, 64), (32, 64, 64),
    # 
    (32, 64, 128), (32, 64, 128), (32, 64, 128),
    # 
    (64, 64, 128), (64, 64, 128), (64, 64, 128),
    # 
    (64, 128, 128), (64, 128, 128), (128, 256, 256), (128, 256, 256)
]


mask_types = (
    ['custom'] + ['checkerboard'] * 4 +                     
    (['channel'] * 3 + ['checkerboard'] * 3) * 2 +  
    ['channel'] * 3 +                          
    ['checkerboard'] * 4                      
)

#mask_types = (
#    ['checkerboard'] + ['checkerboard'] * 4 +                     
#    (['channel'] * 3 + ['checkerboard'] * 3) * 2 +  
#    ['channel'] * 3 +                          
#    ['checkerboard'] * 4                      
#)

real_nvp = CondMSRealNVP(
        num_coupling_layers=24,
        num_classes=NUM_CLASSES,
        input_shape=(1, 184, 184),
        squeeze_layers=[5, 11, 17],
        split_layers=[7, 13, 19],
        layer_filters=filters_per_layer,
        mask_types=mask_types,
        verbose = False
    )

# Assign to model the custom mask
mask_path = os.path.join(os.path.dirname(base_path), 'custom_png/coronal_slice_110_rotated.png') 
print(mask_path) 

# Load image as float tensor
mask_img = Image.open(mask_path).convert('L')
mask_np = np.array(mask_img)  # shape (H, W)
mask_tensor = torch.from_numpy(mask_np).float() / 255.0  # values 0.0 or 1.0
real_nvp.custom_mask_tensor = mask_tensor  # (H, W)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
real_nvp = real_nvp.to(device)


# Define the optimizer and scheduler
optimizer = torch.optim.Adam(real_nvp.parameters(), lr=1e-3)
n_epochs = 200
scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-5)

/home/almalinux/data/jupyter_notebooks/cNF_folder/MAGIC-Flow/custom_png/coronal_slice_110_rotated.png


## Load Checkpoint

In [4]:
# Load the checkpoint
checkpoint = torch.load(os.path.join(model_folder, "best_checkpoint_modality.pth"), map_location=device)

# Load model/optimizer/scheduler states
real_nvp.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

# Optionally resume training stats
start_epoch = checkpoint.get("epoch", 0)
global_step = checkpoint.get("global_step", 0)
history = checkpoint.get("history", {"loss": [], "val_loss": []})

print(f"Loaded model from epoch {start_epoch}, step {global_step}")

Loaded model from epoch 190, step 74670


## Generate Samples

In [5]:
def generate_samples_for_class(model, num_samples=5, class_id=None, temperature=1.0, device='cuda',
                               closer_view=False, save_as_pdf=False, pdf_filename="generated_images.pdf"):

    model.eval()

    z_samples = model.distribution.sample((num_samples,)).to(device) * temperature

    if class_id is not None:
        y_samples = torch.zeros((num_samples, model.num_classes), device=device)
        y_samples[:, class_id] = 1
    else:
        y_samples = torch.eye(model.num_classes, device=device).repeat(num_samples // model.num_classes + 1, 1)[:num_samples]

    with torch.no_grad():
        generated_images = model(z_samples, y_samples, reverse=True)

    generated_images = generated_images * 0.5 + 0.5
    generated_images = torch.clamp(generated_images, 0, 1)
    generated_images = generated_images.cpu().squeeze().numpy()

    # Determine layout: max 6 columns
    max_cols = 6
    cols = min(num_samples, max_cols)
    rows = math.ceil(num_samples / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))

    # Flatten axes if 2D
    if rows > 1 or cols > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    for i in range(num_samples):
        img = generated_images[i]
        img = np.rot90(img)

        # Center crop to 182x182
        h, w = img.shape[:2]
        top = (h - 182) // 2
        left = (w - 182) // 2
        img = img[top:top + 182, left:left + 182]

        ax = axes[i]
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.axis('off')

        if not closer_view:
            label = f'Class: {class_id}' if class_id is not None else f'Class: {torch.argmax(y_samples[i]).item()}'
            ax.set_title(label)

    # Turn off extra axes if num_samples < rows*cols
    for j in range(num_samples, len(axes)):
        axes[j].axis('off')

    if closer_view:
        plt.subplots_adjust(wspace=0, hspace=0, left=0, right=1, top=1, bottom=0)
    else:
        plt.tight_layout()

    if save_as_pdf:
        fig.savefig(pdf_filename, format='pdf', bbox_inches='tight', pad_inches=0)

    plt.show()

In [ ]:
generate_samples_for_class(real_nvp, num_samples=24, class_id=2, temperature=1.0, device=device,
                                closer_view=True, save_as_pdf=False, pdf_filename="output_images_mm/t2_gen.pdf")

## Generate and Save Synthetic Data

In [7]:
def generate_samples_for_class(model, num_samples=5, class_id=None, temperature=1.0, device='cuda'):
    
    model.eval()

    z_samples = model.distribution.sample((num_samples,)).to(device) * temperature

    if class_id is not None:
        y_samples = torch.zeros((num_samples, model.num_classes), device=device)
        y_samples[:, class_id] = 1
    else:
        y_samples = torch.eye(model.num_classes, device=device).repeat(num_samples // model.num_classes + 1, 1)[:num_samples]

    with torch.no_grad():
        generated_images = model(z_samples, y_samples, reverse=True)

    generated_images = generated_images * 0.5 + 0.5
    generated_images = torch.clamp(generated_images, 0, 1)
    generated_images = generated_images.cpu().squeeze().numpy()

    return generated_images


def generate_samples_in_batches(model, total_samples, batch_size, class_id=None, temperature=1.0, device='cuda'):
    """
    Generate synthetic images in batches to avoid memory overload.
    
    Args:
        model: Your trained generative model.
        total_samples (int): Total number of images to generate.
        batch_size (int): Number of images per batch.
        class_id (int or None): Class to condition on, if applicable.
        temperature (float): Sampling temperature.
        device (str): 'cuda' or 'cpu'.
    
    Returns:
        List of numpy arrays: Generated images.
    """
    all_generated = []
    remaining = total_samples
    while remaining > 0:
        current_batch_size = min(batch_size, remaining)
        batch_images = generate_samples_for_class(
            model, 
            num_samples=current_batch_size, 
            class_id=class_id, 
            temperature=temperature, 
            device=device
        )
        all_generated.extend(batch_images)
        remaining -= current_batch_size
        print(f"Generated {total_samples - remaining}/{total_samples} images", end='\r')

    return all_generated

def generate_and_save_images(model, device, class_names, num_classes, samples_per_class, batch_size, save_dir, crop_size=(182, 182)):
    """
    Generates synthetic images, crops them to a specified size, and saves them in corresponding class folders.

    Parameters:
    - model: Trained model to generate images (e.g., a GAN or RealNVP model).
    - device: Device (CPU or CUDA).
    - class_names: List of class names.
    - num_classes: Number of classes.
    - samples_per_class: Number of samples to generate for each class.
    - batch_size: Number of images to generate per batch.
    - save_dir: Directory where generated images will be saved.
    - crop_size: Tuple (width, height) specifying the crop dimensions (default is 182x182).
    """
    # Ensure the save directory exists
    os.makedirs(save_dir, exist_ok=True)
    
    for class_id in range(num_classes):
        print(f"Generating images for class {class_names[class_id]}")

        # Create a subfolder for the class
        class_save_path = os.path.join(save_dir, class_names[class_id])
        os.makedirs(class_save_path, exist_ok=True)

        # Generate synthetic images
        gen_imgs = generate_samples_in_batches(
            model=model,
            total_samples=samples_per_class,
            batch_size=batch_size,
            class_id=class_id,
            temperature=1.0,
            device=device
        )

        # Crop and save each generated image
        for i, img in enumerate(gen_imgs):
            img_tensor = torch.tensor(img).to(device)  
            img_pil = transforms.ToPILImage()(img_tensor.cpu())  
            
            # Crop image to the desired size
            width, height = img_pil.size
            left = (width - crop_size[0]) // 2
            top = (height - crop_size[1]) // 2
            right = (width + crop_size[0]) // 2
            bottom = (height + crop_size[1]) // 2
            img_cropped = img_pil.crop((left, top, right, bottom))

            # Save the cropped image
            img_cropped.save(os.path.join(class_save_path, f"generated_{i + 1}.png"))

        print(f"Saved {samples_per_class} images for class {class_names[class_id]} in {class_save_path}")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
class_names = ['mri_flair', 'mri_t1w', 'mri_t2w', 'pet_amyloid', 'pet_fdg', 'pet_tau']
num_classes = len(class_names)
samples_per_class = 50
batch_size = 25
save_dir =  os.path.join(base_path, "synthetic_dataset")  
generate_and_save_images(real_nvp, device, class_names, num_classes, samples_per_class, batch_size, save_dir)